In [11]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from langchain_core.output_parsers import StrOutputParser

from langgraph.checkpoint.memory import InMemorySaver

In [10]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [12]:
# create a state

class LLMQA(TypedDict):

    topic : str
    joke : str
    explanation : str

In [13]:
# define function

def llm_joke(state : LLMQA) -> LLMQA:

    que = state['topic']
    prompt = f"give a one liner joke on this topic : {que}"

    ans = llm.invoke(prompt)

    parser = StrOutputParser()
    clean_string = parser.invoke(ans)

    return {"joke": clean_string}


def llm_explain(state: LLMQA) -> LLMQA:

    joke = state['joke']
    prompt = f"explain me this joke in just 1-2 line : {joke}"
    
    ex = llm.invoke(prompt)

    parser = StrOutputParser()
    clean_string = parser.invoke(ex)
    
    return {"explanation": clean_string}


In [28]:
# define your graph
graph = StateGraph(LLMQA)

# add nodes to your graph
graph.add_node("llm_joke",llm_joke)
graph.add_node("llm_explain",llm_explain)

# add edges to your graph
graph.add_edge(START,"llm_joke")
graph.add_edge("llm_joke","llm_explain")
graph.add_edge("llm_explain",END)

# define check pointer
checkpointer = InMemorySaver()

# compile the graph
workflow = graph.compile(checkpointer=checkpointer)


In [29]:
config1 = {"configurable" : {"thread_id" : "1"}}

query = input("Enter the topic :: ")
initial_state = {"topic" : query}

final_state = workflow.invoke(initial_state, config = config1)

print(final_state)

{'topic': 'ai', 'joke': 'Why did the AI quit its job? It said it was tired of being *prompted*.', 'explanation': 'It\'s a pun! AI models *literally* need "prompts" (instructions) to operate, but in human terms, being constantly "prompted" means being told what to do, which is annoying.'}


In [30]:
config2 = {"configurable" : {"thread_id" : "2"}}

query = input("Enter the topic :: ")
initial_state = {"topic" : query}

final_state = workflow.invoke(initial_state, config = config2)

print(final_state)

{'topic': 'water', 'joke': "I tried to tell a joke about water, but it didn't hold water.", 'explanation': 'It\'s a pun! The joke about water was illogical or unsound (idiomatically "didn\'t hold water"), contrasting with water\'s literal ability to be held.'}


In [32]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'water', 'joke': "I tried to tell a joke about water, but it didn't hold water.", 'explanation': 'It\'s a pun! The joke about water was illogical or unsound (idiomatically "didn\'t hold water"), contrasting with water\'s literal ability to be held.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b3af-b8da-60de-8002-11f9477d281c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-29T10:48:06.992279+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b3af-8d1a-6ddc-8001-64bb391a1d5a'}}, tasks=(), interrupts=())

In [34]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'water', 'joke': "I tried to tell a joke about water, but it didn't hold water.", 'explanation': 'It\'s a pun! The joke about water was illogical or unsound (idiomatically "didn\'t hold water"), contrasting with water\'s literal ability to be held.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b3af-b8da-60de-8002-11f9477d281c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-29T10:48:06.992279+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b3af-8d1a-6ddc-8001-64bb391a1d5a'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'water', 'joke': "I tried to tell a joke about water, but it didn't hold water."}, next=('llm_explain',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b3af-8d1a-6ddc-8001-64bb391a1d5a'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at

In [35]:
workflow.invoke(None , {'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b3af-0d6c-6dd3-8000-1f0806dbc0dd'}})

{'topic': 'water',
 'joke': "Water is always up-to-date on the news because it's constantly dealing with *current* events.",
 'explanation': 'The joke is a pun on the word "current." "Current events" means up-to-date news, but "currents" also refer to moving bodies of water.'}

In [36]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'water', 'joke': "Water is always up-to-date on the news because it's constantly dealing with *current* events.", 'explanation': 'The joke is a pun on the word "current." "Current events" means up-to-date news, but "currents" also refer to moving bodies of water.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b40d-5ffe-61f7-8003-5d257fd1d5de'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-29T11:30:00.968017+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b40d-3b4c-61bd-8002-902b17d735d9'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'water', 'joke': "Water is always up-to-date on the news because it's constantly dealing with *current* events."}, next=('llm_explain',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18b40d-3b4c-61bd-8002-902b17d735d9'}}, metadata={'source'